# Cross-Country Comparison: Solar Radiation Analysis
## Benin, Sierra Leone, and Togo

This notebook synthesizes the cleaned datasets from three countries to:
- Compare solar radiation metrics (GHI, DNI, DHI)
- Identify relative solar potential
- Perform statistical testing
- Provide actionable insights

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

## 1. Load Cleaned Datasets

In [ ]:
# Robust path detection
import os

def find_data_file(filename):
    """Find data file in various possible locations"""
    possible_paths = [
        f"./data/{filename}",
        f"../data/{filename}",
        f"data/{filename}"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    raise FileNotFoundError(f"Could not find {filename} in any of: {possible_paths}")

# Load cleaned datasets
countries = {
    'Benin': 'benin_clean.csv',
    'Sierra Leone': 'sierraleone_clean.csv',
    'Togo': 'togo_clean.csv'
}

dataframes = {}

for country, filename in countries.items():
    try:
        file_path = find_data_file(filename)
        df = pd.read_csv(file_path)
        
        # Parse timestamp if exists
        if 'Timestamp' in df.columns:
            df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        
        df['Country'] = country
        dataframes[country] = df
        print(f"✅ Loaded {country}: {df.shape[0]:,} records")
    except FileNotFoundError as e:
        print(f"⚠️ Warning: {e}")

print(f"\n📊 Total countries loaded: {len(dataframes)}")

In [ ]:
# Combine all datasets
if len(dataframes) > 0:
    df_combined = pd.concat(dataframes.values(), ignore_index=True)
    print(f"Combined dataset shape: {df_combined.shape}")
    print(f"\nCountries distribution:")
    print(df_combined['Country'].value_counts())
else:
    print("⚠️ No datasets loaded. Please ensure cleaned CSV files exist in data/ directory.")

## 2. Summary Statistics Comparison

In [ ]:
# Create summary table for key metrics
metrics = ['GHI', 'DNI', 'DHI']
summary_data = []

for country in dataframes.keys():
    df = dataframes[country]
    for metric in metrics:
        if metric in df.columns:
            summary_data.append({
                'Country': country,
                'Metric': metric,
                'Mean': df[metric].mean(),
                'Median': df[metric].median(),
                'Std Dev': df[metric].std(),
                'Min': df[metric].min(),
                'Max': df[metric].max()
            })

summary_df = pd.DataFrame(summary_data)

print("="*80)
print("SUMMARY STATISTICS: GHI, DNI, DHI BY COUNTRY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

In [ ]:
# Create pivot table for better visualization
for metric in metrics:
    metric_summary = summary_df[summary_df['Metric'] == metric][['Country', 'Mean', 'Median', 'Std Dev']]
    metric_summary = metric_summary.set_index('Country')
    
    print(f"\n{'='*60}")
    print(f"{metric} Comparison")
    print(f"{'='*60}")
    print(metric_summary.round(2))
    print(f"{'='*60}")

## 3. Visual Comparison: Side-by-Side Boxplots

In [ ]:
# Boxplots for each metric
if len(dataframes) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    for idx, metric in enumerate(metrics):
        if metric in df_combined.columns:
            # Prepare data for boxplot
            data_to_plot = [dataframes[country][metric].dropna() 
                           for country in dataframes.keys() 
                           if metric in dataframes[country].columns]
            
            bp = axes[idx].boxplot(data_to_plot, 
                                   labels=list(dataframes.keys()),
                                   patch_artist=True,
                                   showmeans=True)
            
            # Color the boxes
            for patch, color in zip(bp['boxes'], colors[:len(dataframes)]):
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
            
            axes[idx].set_title(f'{metric} Distribution by Country', fontweight='bold', fontsize=12)
            axes[idx].set_ylabel(f'{metric} (W/m²)' if metric != 'Tamb' else 'Temperature (°C)')
            axes[idx].grid(True, alpha=0.3)
            axes[idx].tick_params(axis='x', rotation=45)
    
    plt.suptitle('Solar Radiation Metrics: Cross-Country Comparison', 
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data available for plotting")

In [ ]:
# Interactive boxplots using Plotly
if len(dataframes) > 0:
    for metric in metrics:
        if metric in df_combined.columns:
            fig = px.box(df_combined, 
                        x='Country', 
                        y=metric,
                        color='Country',
                        title=f'{metric} Distribution by Country',
                        labels={metric: f'{metric} (W/m²)', 'Country': 'Country'},
                        color_discrete_sequence=['#3498db', '#e74c3c', '#2ecc71'])
            
            fig.update_layout(height=500, showlegend=True)
            fig.show()

## 4. Statistical Testing: ANOVA / Kruskal-Wallis

In [ ]:
# Perform statistical tests
print("="*80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*80)

statistical_results = []

for metric in metrics:
    if metric in df_combined.columns:
        # Prepare groups for each country
        groups = [dataframes[country][metric].dropna() 
                 for country in dataframes.keys() 
                 if metric in dataframes[country].columns]
        
        if len(groups) >= 2:
            # Perform Kruskal-Wallis H-test (non-parametric alternative to one-way ANOVA)
            h_stat, p_value_kw = stats.kruskal(*groups)
            
            # Also perform one-way ANOVA (parametric)
            f_stat, p_value_anova = stats.f_oneway(*groups)
            
            # Determine significance
            significance = "***" if p_value_anova < 0.001 else \
                          "**" if p_value_anova < 0.01 else \
                          "*" if p_value_anova < 0.05 else "ns"
            
            statistical_results.append({
                'Metric': metric,
                'ANOVA F-statistic': f_stat,
                'ANOVA p-value': p_value_anova,
                'Kruskal-Wallis H': h_stat,
                'Kruskal-Wallis p-value': p_value_kw,
                'Significance': significance
            })
            
            print(f"\n{metric}:")
            print(f"  ANOVA F-statistic: {f_stat:.2f}, p-value: {p_value_anova:.2e} {significance}")
            print(f"  Kruskal-Wallis H: {h_stat:.2f}, p-value: {p_value_kw:.2e}")
            
            if p_value_anova < 0.05:
                print(f"  ✅ Significant difference detected between countries (p < 0.05)")
            else:
                print(f"  ⚠️ No significant difference between countries (p ≥ 0.05)")

# Create results DataFrame
stats_df = pd.DataFrame(statistical_results)
print(f"\n{'='*80}")
print("SUMMARY TABLE")
print(f"{'='*80}")
print(stats_df.to_string(index=False))
print(f"\n{'='*80}")
print("Significance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print(f"{'='*80}")

## 5. Country Rankings by Average GHI

In [ ]:
# Calculate average GHI per country
if 'GHI' in df_combined.columns:
    ghi_avg = df_combined.groupby('Country')['GHI'].mean().sort_values(ascending=False)
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(ghi_avg.index, ghi_avg.values, 
                  color=['#FFD700', '#C0C0C0', '#CD7F32'][:len(ghi_avg)],
                  edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.2f}\nW/m²',
               ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    ax.set_ylabel('Average GHI (W/m²)', fontsize=12, fontweight='bold')
    ax.set_title('Country Ranking by Average Global Horizontal Irradiance (GHI)', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print("\n🏆 COUNTRY RANKINGS BY AVERAGE GHI:")
    print("="*50)
    for rank, (country, value) in enumerate(ghi_avg.items(), 1):
        medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
        print(f"{medal} #{rank}: {country:15s} - {value:.2f} W/m²")
    print("="*50)

In [ ]:
# Comprehensive ranking table
ranking_data = []

for metric in ['GHI', 'DNI', 'DHI']:
    if metric in df_combined.columns:
        metric_avg = df_combined.groupby('Country')[metric].mean().sort_values(ascending=False)
        for rank, (country, value) in enumerate(metric_avg.items(), 1):
            ranking_data.append({
                'Metric': metric,
                'Rank': rank,
                'Country': country,
                'Average Value': value
            })

ranking_df = pd.DataFrame(ranking_data)

print("\n📊 COMPREHENSIVE METRIC RANKINGS")
print("="*80)
for metric in ['GHI', 'DNI', 'DHI']:
    print(f"\n{metric}:")
    metric_data = ranking_df[ranking_df['Metric'] == metric]
    print(metric_data[['Rank', 'Country', 'Average Value']].to_string(index=False))
print("="*80)

## 6. Additional Insights: Variability Analysis

In [ ]:
# Coefficient of Variation (CV) analysis
print("\n📈 VARIABILITY ANALYSIS (Coefficient of Variation)")
print("="*80)
print("CV = (Standard Deviation / Mean) × 100%")
print("Higher CV indicates greater variability/instability\n")

variability_data = []

for country in dataframes.keys():
    df = dataframes[country]
    for metric in metrics:
        if metric in df.columns:
            mean_val = df[metric].mean()
            std_val = df[metric].std()
            cv = (std_val / mean_val) * 100 if mean_val != 0 else 0
            
            variability_data.append({
                'Country': country,
                'Metric': metric,
                'CV (%)': cv
            })

variability_df = pd.DataFrame(variability_data)
variability_pivot = variability_df.pivot(index='Country', columns='Metric', values='CV (%)')

print(variability_pivot.round(2))
print("\n" + "="*80)

In [ ]:
# Visualize variability
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(dataframes))
width = 0.25

for i, metric in enumerate(metrics):
    cv_values = [variability_pivot.loc[country, metric] 
                for country in variability_pivot.index 
                if metric in variability_pivot.columns]
    ax.bar(x + i*width, cv_values, width, label=metric)

ax.set_xlabel('Country', fontweight='bold')
ax.set_ylabel('Coefficient of Variation (%)', fontweight='bold')
ax.set_title('Data Variability by Country and Metric', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(variability_pivot.index)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Key Observations & Insights

### Summary of Cross-Country Analysis:

**Key Findings:**

1. **Solar Potential Ranking**: Based on average GHI values, the countries can be ranked for their solar energy potential. The statistical tests (ANOVA/Kruskal-Wallis) indicate whether these differences are statistically significant.

2. **Variability Considerations**: Countries with higher coefficient of variation (CV) show greater day-to-day variability in solar radiation, which may impact the reliability of solar installations and require more robust energy storage solutions.

3. **Metric-Specific Insights**: 
   - **GHI** represents total solar radiation and is most relevant for flat-plate collectors
   - **DNI** is crucial for concentrated solar power (CSP) systems
   - **DHI** indicates diffuse radiation, important for understanding cloud cover effects

### Actionable Recommendations:

- Countries with **highest mean GHI/DNI** offer best potential for solar installations
- Countries with **lowest variability (CV)** provide more predictable and stable solar resource
- **Statistical significance** (p-values) confirms whether observed differences are reliable or due to chance

### Next Steps:

1. Consider seasonal variations in detailed analysis
2. Factor in economic and infrastructure considerations
3. Assess grid integration capabilities for each region
4. Evaluate land availability and environmental impact

In [ ]:
# Generate final summary report
print("\n" + "="*80)
print("FINAL SUMMARY REPORT: CROSS-COUNTRY COMPARISON")
print("="*80)

if len(dataframes) > 0:
    print(f"\n📊 Analysis Coverage:")
    print(f"   - Countries analyzed: {len(dataframes)}")
    print(f"   - Total records: {df_combined.shape[0]:,}")
    print(f"   - Metrics compared: {', '.join(metrics)}")
    
    if 'GHI' in df_combined.columns:
        print(f"\n🏆 Top Country (by avg GHI): {ghi_avg.index[0]} ({ghi_avg.values[0]:.2f} W/m²)")
    
    print(f"\n📈 Statistical Testing:")
    if len(statistical_results) > 0:
        sig_count = sum(1 for r in statistical_results if r['ANOVA p-value'] < 0.05)
        print(f"   - {sig_count}/{len(statistical_results)} metrics show significant differences (p < 0.05)")
    
    print("\n✅ Analysis completed successfully!")
else:
    print("\n⚠️ No data available for analysis")

print("="*80)